# Step 5: Before vs After 비교

Fine-tuning 전후 성능을 시각적으로 비교합니다.

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '5_comparison'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 결과 로드 (각 노트북 디렉토리에서)
before_path = PROJECT_ROOT / '2_before_evaluation' / 'before_results.json'
after_path = PROJECT_ROOT / '4_after_evaluation' / 'after_results.json'

print(f"Before 결과: {before_path}")
print(f"After 결과: {after_path}")

with open(before_path, 'r') as f:
    before = json.load(f)
with open(after_path, 'r') as f:
    after = json.load(f)
    
print("결과 로드 완료!")

In [ ]:
# 비교 테이블 출력
print("="*70)
print("          📊 Fine-tuning 효과 비교 (한국인 딥페이크 테스트)")
print("="*70)
print(f"{'Metric':<15} {'Before (FF++)':<20} {'After (KoDF)':<20} {'개선':<15}")
print("-"*70)

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
for m in metrics:
    b, a = before[m]*100, after[m]*100
    diff = a - b
    print(f"{m.capitalize():<15} {b:>17.1f}%   {a:>17.1f}%   {'+' if diff>0 else ''}{diff:>12.1f}%p")

print("="*70)

In [ ]:
# 막대 그래프 비교
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics))
width = 0.35

before_vals = [before[m]*100 for m in metrics]
after_vals = [after[m]*100 for m in metrics]

bars1 = ax.bar(x - width/2, before_vals, width, label='Before (FF++ Pretrained)', color='#ff6b6b')
bars2 = ax.bar(x + width/2, after_vals, width, label='After (KoDF Fine-tuned)', color='#4ecdc4')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Fine-tuning 효과 비교\n(한국인 딥페이크 테스트 데이터)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics])
ax.legend()
ax.set_ylim(0, 100)
ax.axhline(y=70, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=90, color='gray', linestyle='--', alpha=0.5)

for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150)
plt.show()

In [ ]:
# 결론 출력
improvement = after['accuracy']*100 - before['accuracy']*100
print("\n" + "="*70)
print("                         🎯 결론")
print("="*70)
print(f"""
  FaceForensics++ pretrained 모델은 서양인 얼굴 위주로 학습되어
  한국인 딥페이크 탐지에서 약 {before['accuracy']*100:.0f}%의 정확도를 보였습니다.
  
  KoDF(한국인 딥페이크) 데이터로 Fine-tuning 후
  정확도가 약 {after['accuracy']*100:.0f}%로 향상되었습니다.
  
  ✅ Fine-tuning으로 +{improvement:.1f}%p 성능 개선!
""")
print("="*70)